In [1]:
import sys
from pathlib import Path

parent_folder = Path.cwd().parent
if str(parent_folder) not in sys.path:
    sys.path.append(str(parent_folder))
    
import warnings
warnings.filterwarnings('ignore')
import numpy as np

import simulation as sim
import xobjects as xo
import xtrack as xt
import copy

In [2]:
gemitt = 60e-6
n_turns=200
n_particles=10000

line = sim.build_line(folder=Path('../Lattice_Files/02_Aperture_Lattice_res/'))

ref = line.particle_ref
bg = ref.beta0 * ref.gamma0

tw = line.twiss(method='4d', freeze_longitudinal=True)
particles_base = line.build_particles(
    x_norm=np.random.normal(size=n_particles),
    y_norm=np.random.normal(size=n_particles),
    px_norm=np.random.normal(size=n_particles),
    py_norm=np.random.normal(size=n_particles),
    nemitt_x=bg*gemitt,
    nemitt_y=bg*gemitt,
    method='4d',
    freeze_longitudinal=True, mode='normalized_transverse')

particles_base.x += (tw.x[0] - np.mean(particles_base.x))
particles_base.y += (tw.y[0] - np.mean(particles_base.y))
particles_base.px += (tw.px[0] - np.mean(particles_base.px))
particles_base.py += (tw.py[0] - np.mean(particles_base.py))

Converting sequence "synchrotron":   0%|          | 0/1990 [00:00<?, ?it/s]

Compiling ContextCpu kernels...
Done compiling ContextCpu kernels.


In [3]:
line = sim.build_line(folder=Path('../Lattice_Files/02_Aperture_Lattice_res/'))
            
circumference = line.get_length()
s_positions = line.get_s_elements()
aperture_classes = (xt.LimitRect, xt.LimitEllipse, xt.LimitRectEllipse, xt.LimitPolygon)
amplitude = 0.01  # 2cm amplitude

for i, name in enumerate(line.element_names):
    element = line[name]
    if isinstance(element, aperture_classes):
        s = s_positions[i]
        shift_x = amplitude * np.sin(13 * 2 * np.pi * s / circumference)
        element.min_x += shift_x
        element.max_x += shift_x


particles = copy.deepcopy(particles_base)

line = sim.add_monitors(line, particles, n_monitors=1, n_turns=n_turns)

twiss = line.twiss(method='4d', freeze_longitudinal=True)

line, beam_df, aper_df = sim.add_spacecharge(line, beam_intensity=1e13,
                                             n_interactions=100, 
                                             gemitt=gemitt, indirect=True)

line.build_tracker(_context=xo.ContextCpu(omp_num_threads=10))
line.track(particles, num_turns=n_turns, time=True)

sim.save(line, particles, twiss, filename=f'data/wiggle_matched_fp', aper_df=aper_df, beam_df=beam_df)

Converting sequence "synchrotron":   0%|          | 0/1990 [00:00<?, ?it/s]

Slicing line:   0%|          | 0/3980 [00:00<?, ?it/s]

Compiling ContextCpu kernels...
Done compiling ContextCpu kernels.


Slicing line:   0%|          | 0/3981 [00:00<?, ?it/s]

Compiling ContextCpu kernels...
Done compiling ContextCpu kernels.
Compiling ContextCpu kernels...
Done compiling ContextCpu kernels.
File saved at data/wiggle_matched_fp


In [4]:
line = sim.build_line(folder=Path('../Lattice_Files/02_Aperture_Lattice_res/'))
            
particles = copy.deepcopy(particles_base)

line = sim.add_monitors(line, particles, n_monitors=1, n_turns=n_turns)

twiss = line.twiss(method='4d', freeze_longitudinal=True)

line, beam_df, aper_df = sim.add_spacecharge(line, beam_intensity=1e13,
                                             n_interactions=100, 
                                             gemitt=gemitt, indirect=True)

line.build_tracker(_context=xo.ContextCpu(omp_num_threads=10))
line.track(particles, num_turns=n_turns, time=True)

sim.save(line, particles, twiss, filename=f'data/straight_matched_fp', aper_df=aper_df, beam_df=beam_df)

Converting sequence "synchrotron":   0%|          | 0/1990 [00:00<?, ?it/s]

Slicing line:   0%|          | 0/3980 [00:00<?, ?it/s]

Compiling ContextCpu kernels...
Done compiling ContextCpu kernels.


Slicing line:   0%|          | 0/3981 [00:00<?, ?it/s]

Compiling ContextCpu kernels...
Done compiling ContextCpu kernels.
Compiling ContextCpu kernels...
Done compiling ContextCpu kernels.
File saved at data/straight_matched_fp


In [5]:
line = sim.build_line(folder=Path('../Lattice_Files/02_Aperture_Lattice_res/'))
            
particles = copy.deepcopy(particles_base)

line = sim.add_monitors(line, particles, n_monitors=1, n_turns=n_turns)

twiss = line.twiss(method='4d', freeze_longitudinal=True)

line, beam_df, aper_df = sim.add_spacecharge(line, beam_intensity=1e13,
                                             n_interactions=100, 
                                             gemitt=gemitt, indirect=False)

line.build_tracker(_context=xo.ContextCpu(omp_num_threads=10))
line.track(particles, num_turns=n_turns, time=True)

sim.save(line, particles, twiss, filename=f'data/direct_matched_fp', aper_df=aper_df, beam_df=beam_df)

Converting sequence "synchrotron":   0%|          | 0/1990 [00:00<?, ?it/s]

Slicing line:   0%|          | 0/3980 [00:00<?, ?it/s]

Compiling ContextCpu kernels...
Done compiling ContextCpu kernels.


Slicing line:   0%|          | 0/3981 [00:00<?, ?it/s]

Compiling ContextCpu kernels...
Done compiling ContextCpu kernels.
Compiling ContextCpu kernels...
Done compiling ContextCpu kernels.
File saved at data/direct_matched_fp
